PDF -> TEXT CODE

In [ ]:
# Set the folder path
pdf_folder = ""C:\Users\Aqil Anhein\Desktop\TA\Peraturan Pemerintah""

# Get all PDF files in the folder
pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith('.pdf')]

# Loop through each PDF file
for pdf_file in pdf_files:
    pdf_path = os.path.join(pdf_folder, pdf_file)
    txt_filename = os.path.splitext(pdf_file)[0] + ".txt"
    txt_path = os.path.join(pdf_folder, txt_filename)

    # Read the PDF
    with open(pdf_path, "rb") as f:
        reader = PdfReader(f)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""

    # Save to a .txt file
    with open(txt_path, "w", encoding="utf-8") as txt_file:
        txt_file.write(text)

    print(f"Extracted text from: {pdf_file} -> {txt_filename}")


In [ ]:
pdf_folder = r"C:\Users\Aqil Anhein\Desktop\TA\Undang Undang"

# Create output subfolder
output_folder = os.path.join(pdf_folder, "cleaned_txt")
os.makedirs(output_folder, exist_ok=True)

# Get all PDF files in the folder
pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith('.pdf')]

# Loop through each PDF file
for pdf_file in pdf_files:
    pdf_path = os.path.join(pdf_folder, pdf_file)
    txt_filename = os.path.splitext(pdf_file)[0] + ".txt"
    txt_path = os.path.join(output_folder, txt_filename)  # <-- save in new folder

    # Read the PDF
    with open(pdf_path, "rb") as f:
        reader = PdfReader(f)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""

    # Save to a .txt file in output folder
    with open(txt_path, "w", encoding="utf-8") as txt_file:
        txt_file.write(text)

    print(f"Extracted text from: {pdf_file} -> {txt_filename}")

CLEANING CODE

In [ ]:
def remove_ellipsis_lines(text):
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        # Remove line if it contains '...' or '…' or similar ellipsis patterns
        if re.search(r'\.\.\.|…', line):
            continue
        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

In [ ]:
def remove_footer_clutter(text):
    # Step 1: Merge lines to detect cross-line footer phrases
    text = re.sub("PRESIDEN\nREPUBLIK INDONESIA", "", text)
    # Step 2: Forcefully remove case-sensitive footer parts
    text = re.sub("PRESIDEN", "", text)
    text = re.sub("REPUBLIK INDONESIA", "", text)
    text = re.sub(
        r'[\w.,;:\)\(]*PRESIDEN\s*[\n ]*REPUBLIK INDONESIA',
        '',
        text
    )
    text = re.sub(
        r'\.?\s*PRESIDEN\s*\n\s*REPUBLIK INDONESIA',
        '',
        text
    )
    # Step 3: Remove page numbers like -1-, -23- (exact match on a line)
    #text = re.sub(r'(?m)^-\d+-$', '', text)
    text = re.sub(r'(?m)^\s*-\s*\d+\s*-\s*$', '', text)
    # Step 4: Remove URLs
    text = re.sub(r'^.*(https?://\S+|www\.\S+).*\n?', '', text, flags=re.MULTILINE)
    return text

In [ ]:
def normalize_and_extract_memutuskan(text):
    # Step 1: Normalize variants like MEMUTU SKAN, MEMUTVSKAN, etc.
    pattern = r'\bM[EI1]{1,2}M?U[T7]U[S5]?[ \n]*K[AI1]N\b'
    text = re.sub(pattern, 'MEMUTUSKAN', text, flags=re.IGNORECASE)

    # Step 2: Extract everything from MEMUTUSKAN onward
    split = text.split('MEMUTUSKAN', 1)
    if len(split) > 1:
        return 'MEMUTUSKAN' + split[1]
    else:
        print("[WARN] MEMUTUSKAN not found after normalization.")
        return text

In [ ]:
def merge_bab_titles(text):
    lines = text.splitlines()
    merged_lines = []
    i = 0
    while i < len(lines):
        line = lines[i].strip()

        # Match lines that are only "BAB IX", "BAB V", etc. — nothing else
        if re.fullmatch(r'BAB\s+[IVXLCDM]+', line, flags=re.IGNORECASE):
            # Next line should be the title
            if i + 1 < len(lines):
                title_line = lines[i + 1].strip()
                # Merge and force to uppercase
                merged_lines.append(f"{line.upper()} : {title_line}")
                i += 2  # Skip next line
                continue

        # Otherwise, keep the line as-is
        merged_lines.append(lines[i])
        i += 1
    return "\n".join(merged_lines)

In [ ]:
def uppercase_pasal_headers(text):
    lines = text.splitlines()
    updated_lines = []

    for line in lines:
        stripped = line.strip()

        # Check if this line starts with 'Pasal' and is not part of a sentence
        if re.match(r'^Pasal\b.*$', stripped, flags=re.IGNORECASE):
            # Must be a standalone line, not embedded in a paragraph
            updated_lines.append(stripped.upper())
        else:
            updated_lines.append(line)

    return "\n".join(updated_lines)

In [ ]:
def remove_after_first_signature_block(text):
    # Signature start patterns
    pattern = r'(?:Ditetapkan|Disahkan|Diundangkan)\s+di\s+Jakarta|Agar setiap orang mengetahuinya|di\s+Jakarta'

    match = re.search(pattern, text, flags=re.IGNORECASE)

    if not match:
        return text  # no signature block found

    return text[:match.start()].strip()

In [ ]:
# Compile patterns once
MAIN_PATTERN = re.compile(
    r'\b(M|IVI|IM|I\W?I)?[EI1]{1,2}M?U[VvT7]U[S5]?[ \n]*K[AI1]N[:：．. ]?', 
    flags=re.IGNORECASE
)
FALLBACK_PATTERN = re.compile(
    r'[\.\-–—…]*[ \n]*UTUSKAN[:：．. ]?', 
    flags=re.IGNORECASE
)
SPACED_LETTERS_PATTERN = re.compile(
    r'(?:(?:\b|\s)[Mm][\s]*)'
    r'(?:[Ee][\s]*)'
    r'(?:[Mm][\s]*)'
    r'(?:[Uu][\s]*)'
    r'(?:[Tt][\s]*)'
    r'(?:[Uu][\s]*)'
    r'(?:[Ss][\s]*)'
    r'(?:[Kk][\s]*)'
    r'(?:[Aa][\s]*)'
    r'(?:[Nn])'
    r'(?:[\s:：．.]+)?',
    flags=re.IGNORECASE
)

def normalize_and_extract_memutuskan(text, filename="<unknown>"):
    if not isinstance(text, str):
        print(f"[WARN] Invalid text input in file: {filename}")
        return ""

    # Step 1: Try main pattern
    text, count = MAIN_PATTERN.subn('MEMUTUSKAN', text, count=1)

    # Step 2: Fallback to UTUSKAN variants
    if count == 0:
        text, fallback_count = FALLBACK_PATTERN.subn('MEMUTUSKAN', text, count=1)
        count += fallback_count

    # Step 3: Fallback to spaced letters
    if count == 0:
        text, spaced_count = SPACED_LETTERS_PATTERN.subn('MEMUTUSKAN', text, count=1)
        count += spaced_count

    # Final extraction
    if count == 0:
        print(f"[WARN] MEMUTUSKAN not found in file: {filename}")
        return text

    split = text.split('MEMUTUSKAN', 1)
    if len(split) > 1:
        return 'MEMUTUSKAN' + split[1].strip()
    else:
        print(f"[WARN] Failed to split after MEMUTUSKAN in file: {filename}")
        return text

In [ ]:
def process_txt_files(folder_path):
    # Create output folder
    output_folder = os.path.join(folder_path, "final_cleaned")
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt") and not filename.endswith("_cleaned.txt"):
            file_path = os.path.join(folder_path, filename)
            
            # Read original file
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
            
            # Process content
            cleaned_content = normalize_and_extract_memutuskan(content,file_path)
            cleaned_content = remove_ellipsis_lines(cleaned_content)
            cleaned_content = remove_footer_clutter(cleaned_content)
            cleaned_content = remove_after_first_signature_block(cleaned_content)
            cleaned_content = merge_bab_titles(cleaned_content)
            cleaned_content = uppercase_pasal_headers(cleaned_content)
            
            # Save to new file in output folder
            new_filename = filename.replace('.txt', '_cleaned.txt')
            new_path = os.path.join(output_folder, new_filename)
            
            with open(new_path, 'w', encoding='utf-8') as f:
                f.write(cleaned_content)
            
            print(f"Processed: {filename} -> final_cleaned/{new_filename}")

# Example usage
folder_path = r"C:\Users\Aqil Anhein\Desktop\TA\Undang Undang\cleaned_txt"
process_txt_files(folder_path)